# SOP Monitoring Flow Test

This notebook demonstrates the complete SOP (Standard Operating Procedure) monitoring workflow including:
- File upload
- Vision-Language Model (VLM) inference
- SOP detection and validation
- Performance analysis

## Setup and Imports


In [ ]:
from openai import OpenAI
import os
import time
import cv2
import json
import base64
import requests
import pprint


Follow deployment instructions (to be written...) to run the API server, and then set the below variables.

In [ ]:
_THIS_DIR = os.getcwd()
# test video path
video_path = os.path.join(_THIS_DIR, "test_video_whole_sop_h264.mp4")
vlm_prompt_path = os.path.join(_THIS_DIR, "test_data", "inference_asset", "vlm_prompts.txt")
# For SOP detection
action_json_path = os.path.join(_THIS_DIR, "test_data", "actions.json")
# Create OpenAI client pointing to your API server
base_url = "http://localhost:8080/v1"
client = OpenAI(base_url=base_url, api_key="don't care.", max_retries=1)

## Step 1: File Upload

Upload the test video file to the API server for processing.

In [ ]:
print("\n=== Step 1: File Upload ===")
upload_start_time = time.time()
with open(video_path, "rb") as f:
    uploaded_file = client.files.create(
        file=f,
        purpose="vision"
    )
upload_end_time = time.time()
upload_duration = upload_end_time - upload_start_time

print("File uploaded successfully!")
print(f"File ID: {uploaded_file.id}")
print(f"File size: {uploaded_file.bytes} bytes")
print(f"Upload time: {upload_duration:.2f} seconds")

## Step 2: Chat Completions with Uploaded File

Send the uploaded video to the Vision-Language Model (VLM) for analysis using the predefined prompts.

In [ ]:
print("\n=== Step 2: Chat completions with uploaded file ===")
with open(vlm_prompt_path, "r") as f:
    prompt = f.read()

chat_start_time = time.time()
chat_response = client.chat.completions.create(
    model="placeholder", # temporary model name
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                },
                {
                    "type": "image_file",
                    "image_file": {
                        "file_id": uploaded_file.id
                    }
                }
            ]
        }
    ],
)
chat_end_time = time.time()
chat_duration = chat_end_time - chat_start_time

print("Chat completion successful!")
print(f"Chat API processing time: {chat_duration:.2f} seconds")
print(f"Response ID: {chat_response.id}")
print(f"Model: {chat_response.model}")


## Step 3: SOP Sequence Detection

Analyze the VLM output to detect SOP sequences and generate a finite state machine (FSM) visualization.

In [ ]:
print("\n=== Step 3: Check the sop sequence output by VLM ===")
vlm_output = chat_response.choices[0].message.content
with open(action_json_path, "r") as f:
    action_json = json.load(f)

sop_detection_request = {
    "action_json": json.dumps(action_json),
    "vlm_output": vlm_output,
}

# POST to /v1/sop/detection
sop_detection_url = f"{base_url}/sop/detection"
sop_detection_start_time = time.time()
sop_detection_response = requests.post(sop_detection_url, json=sop_detection_request)
sop_detection_end_time = time.time()
sop_detection_duration = sop_detection_end_time - sop_detection_start_time

if sop_detection_response.status_code == 200:
    sop_detection_result = sop_detection_response.json()
    print("\n=== SOP Detection Result ===")
    print(f"Checker ID: {sop_detection_result.get('checker_id')}")
    print(f"{vlm_output}")
    print( f"{pprint.pformat(sop_detection_result)}", flush=True)
else:
    print(f"Error in SOP detection: {sop_detection_response.status_code} {sop_detection_response.text}")
